# System Demo

The following notebook offers a live demo of the system.

The user can chose between three operating modes:
1. Enrollment
2. Identification
3. Verification

For each of the following modes the liveness detection module can be kept active of left unactive

In [ ]:
import os
import torch
import time

# Custom libraries
import utilities as utils
import liveness_detector as liveness_detector
import live_system_demo as live_demo
import face_quality as face_quality
import face_detection as detector
%load_ext autoreload
%autoreload 2

## Config

In [ ]:
Tf = 0.5000
Ti = 0.6833
Tv = 0.6013
img_quality_threshold = 20.264


gallery_name = "gallery.npy"
MODEL_DIR = "Models/"
DATABASE_DIR = "System_Database/gallery"
TESTING_DIR = "Test"
batch_size = 32
DETECTOR = 'MTCNN'       
model = 'InceptionResnetV1-VGGFace2'
device = 'cuda' if torch.cuda.is_available() else 'cpu'
liveness_model = liveness_detector.LivenessMobileNet().to(device) 
mag_face_model_name = 'magface_iresnet50_MS1MV2_ddp_fp32.pth'
liveness_model_name = "liveness_model.pth"

if os.path.exists(os.path.join(MODEL_DIR, liveness_model_name)):
        liveness_model.load_state_dict(torch.load(os.path.join(MODEL_DIR, liveness_model_name)))
else:
        print(f"Liveness model not found. Please ensure '{liveness_model_name}' is in the models directory.")
        
if os.path.exists(os.path.join(MODEL_DIR, mag_face_model_name)):
        magface = face_quality.MagFaceEvaluator(model_path=os.path.join(MODEL_DIR, mag_face_model_name), architecture='iresnet50')
else:
        print(f"MagFace model not found. Please ensure '{mag_face_model_name}' is in the models directory.")

## User Config

In [ ]:
MODE = "Identification"               # Options: "Enrollment", "Identification", "Verification"
USE_LIVENESS = True                 # Options: True, False
USER_NAME = "new_user"             # For Enrollment
CLAIMED_ID = 10                     # For Verification (The integer ID you claim to be)
DATA_DIR = os.path.join(TESTING_DIR, USER_NAME)

## Operating Flow


The system processes data through a sequential pipeline consisting of three main stages: Liveness Detection, Quality Evaluation & Feature Extraction, and specific Operation Mode execution.


### Liveness Detection
This phase is conditional and executes only if the system is configured to check for presentation attacks (USE_LIVENESS = True).

- Input: The raw demo data downloaded by the system.

- Process: The system evaluates the liveness of the provided samples using a specified detector and liveness model.

- Outcome:

Images classified as "fake" (spoofing attempts) are immediately dropped from the pipeline.
Only validated "live" images are passed to the next stage (processed_filtered_test).
If disabled, this stage is skipped, and raw data is passed directly to the quality evaluator.


## Image Quality & Features Extraction

In this phase, the system processes the filtered images to generate a usable biometric template.

- Input: The "live" images from the previous step.

- Process:
Quality Assessment: The system employs MagFace to calculate a quality score for each face, comparing it against a predefined threshold (img_quality_threshold).
The system attempts to select the highest-quality image. If no images meet the strict quality standards, the system falls back to keeping the last available image to ensure continuity.
The selected face is then normalized, and its feature vector is extracted.

- Outcome: A single, normalized embedding (embedding_norm) is produced for use in the matching phase.


## Matching & Execution Mode

The final phase executes one of three specific operating modes based on the MODE configuration. The generated embedding is processed as follows:

#### Enrolment
During this phase a new user can be enrolled in the system, if it's not already present

#### Verification

During this phase, the system evaluates whether a probe image truly corresponds to a claimed identity. The comparison is strictly 1:1: the probe embedding is matched only against the template of the declared subject. Based on the similarity score and a predefined threshold, the system either accepts the claim (genuine match) or rejects it (impostor attempt). 

#### Identification

During this phase, the system determines which identity in the gallery best matches the probe image. Unlike verification, identification operates in a 1:N setting: the probe embedding is compared against all stored templates, and the system selects the identity with the highest similarity (or lowest distance). The output is therefore a predicted subject rather than an accept/reject decision. 



In [ ]:
# Start timer
t0 = time.time()

# Downlaoad Data
test_data = utils.download_demo_data(DATA_DIR)      
utils.check_dataset(n_people_to_display=1, dataset=test_data, target_names=[USER_NAME])

if USE_LIVENESS:
    processed_filtered_test, filtered_test = live_demo.verify_liveness( detector_method=DETECTOR, 
                                                                        data_dir=DATA_DIR, 
                                                                        batch_size=batch_size,
                                                                        liveness_model=liveness_model, 
                                                                        device=device, 
                                                                        Tf=Tf
                                                                        )
else:
    print("\nLiveness Detection is disabled.")
    processed_filtered_test = detector.detect_faces(test_data, method=DETECTOR, verbose=True, cnn=False)  
    filtered_test = test_data

embedding_norm = live_demo.check_faces_and_extract_embedding(processed_filtered_test, filtered_test, model, device, batch_size, magface, img_quality_threshold)


if MODE == "Enrollment":
    try: 
        live_demo.enroll_new_user(new_embedding=embedding_norm, new_name=USER_NAME, gallery_path=gallery_name, database_dir=DATABASE_DIR)
    except Exception as e:
        print(f"Enrollment error: {e}") 
        
elif MODE == "Verification":
    live_demo.verify_user(embedding_norm=embedding_norm, Tv=Tv, claimed_id=CLAIMED_ID, gallery_name=gallery_name, database_dir=DATABASE_DIR)

elif MODE == "Identification":
    live_demo.identify_user(embedding_norm=embedding_norm, Ti=Ti, gallery_name=gallery_name, database_dir=DATABASE_DIR)

else:
    print(f"Invalid MODE: {MODE}. Please choose from 'Enrollment', 'Verification', or 'Identification'.")
    
# End timer
print(f"\nTotal execution time: {time.time() - t0:.2f} seconds")